# BookLens — Sentiment Analysis and Theme Discovery in Books

This notebook runs the full experiment: loading and cleaning the book/review data,
sentiment inference with a pretrained Transformer, a gold-set evaluation, a TF-IDF
baseline for comparison, topic discovery with embeddings → UMAP → HDBSCAN →
BERTopic, and a final join of sentiment and topic results.

Reusable pipeline logic (data loading/cleaning, sentiment inference, the TF-IDF
baseline, the topic modeling pipeline, and the final Pandas join) lives in the
`src/` package and is imported below. Config values (model names, paths, seeds,
batch size, thresholds) live in `src/config.py`.

This notebook holds the environment setup, the pipeline calls, all display/plot
calls, and the hand-written interpretation (topic labels, error-pattern notes,
sensitivity-check conclusion).


## 0. Environment Setup

Repo clone, Kaggle authentication, and the data download stay as one-off manual
setup steps (they need a live Colab runtime/secrets) rather than being wrapped in
`src` functions — see the note at the top of `src/data.py`.

In [ ]:
# Clone the project repo (needs a GH_TOKEN Colab secret with repo access)
from google.colab import userdata
gh_token = userdata.get('GH_TOKEN')


!git clone https://{gh_token}@github.com/arsham05/booklens.git
%cd booklens

In [ ]:
# Kaggle CLI reads these two env vars for authentication
from google.colab import userdata
import os

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

### ⚠️ Kaggle API Authentication Required
To execute the following data acquisition cell, you need a Kaggle API token.

**Instructions:**
1. Generate an API token from your Kaggle account settings (`kaggle.json`).
2. Add your credentials to Google Colab Secrets (the 🔑 icon on the left sidebar).
3. Create two secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY` with your respective details.
4. Ensure "Notebook access" is toggled ON for both secrets.

In [ ]:
# Download the pre-cleaned Kaggle CSV mirror and unzip it into data/
!kaggle datasets download -d mohamedbakhet/amazon-books-reviews --force
!unzip -o amazon-books-reviews.zip -d data/

In [ ]:
# Install packages not preinstalled on the Colab image
!pip install -q bertopic hdbscan umap-learn sentence-transformers

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src import data as data_mod
from src import sentiment as sentiment_mod
from src import baseline as baseline_mod
from src import topics as topics_mod
from src import analysis as analysis_mod

## Data Acquisition, Cleaning & Exploration

In [ ]:
# Load the two raw Kaggle CSV mirror tables, join them, and add a stable book_id
df = data_mod.load_books()
print(f"Merged Dataset Shape: {df.shape}")

In [ ]:
# Draw the manageable working sample (config.N_BOOKS books, config.EXCERPTS_PER_BOOK excerpts each)
sample_df = data_mod.sample_working_set(df)
print(f"Unique books: {sample_df['book_id'].nunique()}")
print(f"Total excerpt rows: {len(sample_df)}")

In [ ]:
# Shape, dtypes, missing values, duplicate titles/summaries, text-length stats
# -- run BEFORE clean_books(), so it reports on the raw sampled data
validation_report = data_mod.validate_books(sample_df)
validation_report

In [ ]:
# Conservative cleaning: collapse whitespace, unescape HTML entities, drop empty
# rows, and add the single-label genre_clean column used everywhere downstream
sample_df = data_mod.clean_books(sample_df)
sample_df.head()

In [ ]:
# Exactly 4 EDA plots, saved to outputs/figures/
eda_plot_paths = data_mod.make_eda_plots(sample_df)
eda_plot_paths

Preview each saved plot inline:

In [ ]:
from PIL import Image

for path in eda_plot_paths:
    display(Image.open(path))

In [ ]:
sample_df.to_csv(config.CLEAN_SAMPLE_PATH, index=False)
print(f"Cleaned dataset saved successfully to '{config.CLEAN_SAMPLE_PATH}'")

## Sentiment Inference

### Sentiment Model Card Details

* **Model Name:** `finiteautomata/bertweet-base-sentiment-analysis`
* **Expected Language:** English
* **Training Domain:** Twitter (Short informal texts / Tweets)
* **Number of Classes:** 3 Sentiment Classes
* **Label Meanings:**
  * `POS` (Positive)
  * `NEG` (Negative)
  * `NEU` (Neutral)

In [ ]:
pipe = sentiment_mod.load_sentiment_pipeline()

In [ ]:
# Smoke test on a handful of excerpts before scaling to the full table
sentiment_mod.predict_sentiment(sample_df["excerpt"][:5].tolist(), pipe)

In [ ]:
import time

# Run inference on every excerpt and time it
start_time = time.time()
sentiment_results = sentiment_mod.predict_sentiment(sample_df["excerpt"].tolist(), pipe)
inference_time = time.time() - start_time

sample_df = sample_df.reset_index(drop=True)
sample_df["sentiment"] = sentiment_results["sentiment"]
sample_df["pos_score"] = sentiment_results["POS_score"]
sample_df["neg_score"] = sentiment_results["NEG_score"]
sample_df["neu_score"] = sentiment_results["NEU_score"]

print(f"Total Excerpts Processed: {len(sample_df)}")
print(f"Inference Time: {inference_time:.2f} seconds")

sample_df.head(10)

### Inference Summary
* **Total Excerpts Processed:** 748
* **Inference Time:** 10.49 seconds
* **Model:** `finiteautomata/bertweet-base-sentiment-analysis`
* **Output Stored:** `sentiment`, `pos_score`, `neg_score`, `neu_score`

(Numbers above are from the original single-row `.apply()` run; `predict_sentiment`
now batches requests via `config.SENTIMENT_BATCH_SIZE`, which returns identical
per-excerpt scores but completes faster — rerun this cell to get current timing.)

## Gold-Set Evaluation

In [ ]:
# Build the ~80-excerpt gold-set candidate sample (balanced across rating buckets
# + extra ambiguous excerpts). This regenerates the exact same rows that were
# hand-labeled into gold_set_annotated.csv, since the three seeds involved are
# fixed in src/config.py -- see the note there before changing any of them.
gold_candidate_df = sentiment_mod.sample_gold_candidates(sample_df)
gold_candidate_df.to_csv(config.GOLD_CANDIDATE_PATH, index=False)
gold_candidate_df["rating_bucket"].value_counts()

In [ ]:
# Load the gold set after it was hand-labeled outside this notebook
gold_df = pd.read_csv(config.GOLD_ANNOTATED_PATH)
gold_df.head()

In [ ]:
sentiment_eval = sentiment_mod.evaluate_sentiment(gold_df["gold_label"], gold_df["sentiment"])

print(f"Macro Precision: {sentiment_eval['macro_precision']:.4f}")
print(f"Macro Recall:    {sentiment_eval['macro_recall']:.4f}")
print(f"Macro F1-Score:  {sentiment_eval['macro_f1']:.4f}")

print("\n" + "="*53 + "\n")

print("Classification Report:")
print(sentiment_eval["classification_report"])

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

display_labels = config.SENTIMENT_DISPLAY_LABELS

# Rows = true (gold) label, columns = predicted label
disp = ConfusionMatrixDisplay(
    confusion_matrix=sentiment_eval["confusion_matrix"], display_labels=display_labels
)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
# Manually inspect the two most common error types from the confusion matrix
df_mislabeled = sentiment_mod.inspect_errors(gold_df)

pd.set_option('display.max_colwidth', None)

for index, row in df_mislabeled.iterrows():
    print(f"Index: {index}")
    print(f"Gold: {row['gold_label']} | Pred: {row['sentiment']}")
    print(f"Scores -> POS: {row['pos_score']:.4f} | NEU: {row['neu_score']:.4f} | NEG: {row['neg_score']:.4f}")
    print(f"Excerpt:\n{row['excerpt']}")
    print("-" * 80)

## Classical Baseline (TF-IDF + Logistic Regression)

In [ ]:
train_df, test_df = baseline_mod.make_gold_split(gold_df)

# Fit only on train, so test-set vocabulary never leaks into the vectorizer
fitted_baseline = baseline_mod.fit_tfidf_baseline(train_df["excerpt"], train_df["gold_label"])

In [ ]:
baseline_predictions = baseline_mod.predict_tfidf_baseline(fitted_baseline, test_df["excerpt"])

comparison = baseline_mod.compare_baseline_vs_transformer(test_df, baseline_predictions)

print("=== TF-IDF + Logistic Regression (Baseline) ===")
print(comparison["baseline"]["classification_report"])

print("\n" + "="*50 + "\n")

# Same held-out set as the baseline above, so the comparison is apples-to-apples
print("=== Pretrained Transformer (on the same test set) ===")
print(comparison["transformer"]["classification_report"])

In [ ]:
pd.set_option('display.max_colwidth', None)

# Cases where the TF-IDF baseline and the Transformer disagree
disagreements = baseline_mod.inspect_disagreements(test_df, baseline_predictions)

print(f"Total disagreements found: {len(disagreements)}")
print("\n--- Inspecting first 5 disagreements ---")

for index, row in disagreements.head(5).iterrows():
    print(f"Index: {index}")
    print(f"Gold Label: {row['gold_label']}")
    print(f"Baseline Predicted: {row['baseline_pred']} | Transformer Predicted: {row['sentiment']}")
    print(f"Excerpt:\n{row['excerpt']}")
    print("-" * 80)

## Preparing Summaries for Topic Discovery

In [ ]:
# One row per book for the topic-modeling branch (a book has one summary,
# unlike the many excerpt rows used for sentiment)
books_df = topics_mod.prepare_documents(sample_df)
documents = books_df["summary"].tolist()

print(f"Total valid unique book summaries: {len(documents)}")
books_df[["book_id", "title", "summary_length"]].head()

## Clustering Pipeline: Embeddings → UMAP → HDBSCAN

In [ ]:
print("Encoding documents...")
embedding_model, embeddings = topics_mod.embed_documents(documents)
print("Successfully generated embeddings!")
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Explicit three-step pipeline: UMAP (5D, for clustering) -> HDBSCAN.
# The fitted umap_model/hdbscan_model are reused as-is inside BERTopic below,
# so the manual pipeline and BERTopic stay aligned.
umap_model, hdbscan_model, cluster_labels = topics_mod.cluster_documents(embeddings)
books_df["cluster"] = cluster_labels

In [ ]:
stats = topics_mod.cluster_stats(cluster_labels)

print(f"Total Clusters Found: {stats['n_clusters']}")
print(f"Total Outliers (Label -1): {stats['n_outliers']}")
print(f"Largest cluster size: {stats['largest_cluster_size']}")
print(f"Smallest cluster size: {stats['smallest_cluster_size']}")

In [ ]:
# Sample a few titles + summaries from the largest clusters plus the outlier
# group -- clusters should never be judged "good" from the 2D plot alone
cluster_inspection_df = topics_mod.inspect_clusters(books_df)

for cluster_id, group in cluster_inspection_df.groupby("cluster_id", sort=False):
    print(f"\n{'='*60}")
    print("CLUSTER -1 (OUTLIERS / NOISE)" if cluster_id == -1 else f"CLUSTER {cluster_id}")
    print(f"{'='*60}")
    for _, row in group.iterrows():
        print(f"Title:   {row['title']}")
        print(f"Summary: {row['summary'][:200]}...")
        print("-" * 40)

## Topic Modeling with BERTopic

In [ ]:
# Reuse the same embedding/UMAP/HDBSCAN objects as the manual pipeline above,
# so this aligns with the cluster labels produced there
topic_model, topics, probs = topics_mod.fit_bertopic(
    documents, embeddings, embedding_model, umap_model, hdbscan_model
)

In [ ]:
topic_model.get_topic_info()

In [ ]:
# Written by hand after reading the representative documents for each topic below --
# if BERTopic is rerun and topic IDs/keywords change, these labels need updating too
human_interpretation = {
    0: "Textbooks, Reference & Self-Help",
    1: "Biographies, Memoirs & Literary History",
    2: "Thrillers, Crime & Suspense Fiction",
    3: "Nature, Ecology & Historical Science",
    4: "War, Espionage & Military History",
    5: "World History, Empires & Politics",
    -1: "Outliers / Uncategorized"
}

final_topic_summary = topics_mod.build_topic_summary(topic_model, books_df, human_interpretation)

display(final_topic_summary[['Topic', 'Count', 'Representation', 'Human_Label', 'Representative_Titles']])

In [ ]:
books_df['Topic'] = topics

outliers = analysis_mod.get_topical_outliers(books_df, topic_col="Topic")
print(f"Total Outliers: {len(outliers)}")
display(outliers[['title', 'summary']].sample(min(3, len(outliers)), random_state=config.INSPECTION_SEED))

### Inspection of Outliers (Topic -1)
A manual inspection of the representative documents in Topic -1 (Outliers) reveals that these summaries are not simply "bad data" or "noise." Instead, they represent highly specific, niche topics within the dataset (e.g., a Southern cookbook, a textbook on healthcare social work, and a historical ethnography of Native American tribes).

Because the dataset is relatively small and the `min_cluster_size` is set to 5, the model could not find enough similar books to form distinct, independent clusters for these unique subjects. Therefore, the outliers here highlight the high diversity of the book catalog rather than a failure of the algorithm. Treating them as valuable information rather than discarding them gives a better picture of the dataset's true distribution.

In [ ]:
print("--- Manual Pipeline (Embeddings + UMAP + HDBSCAN) ---")
print(books_df['cluster'].value_counts().sort_index())

print("\n--- BERTopic ---")

print(books_df['Topic'].value_counts().sort_index())

In [ ]:
crosstab_result = pd.crosstab(books_df['cluster'], books_df['Topic'])
print("--- Cross-tabulation of Document Sets ---")
display(crosstab_result)

### Comparison: Explicit Pipeline vs. BERTopic
As the cross-tabulation (crosstab) table demonstrates, the results of the explicit manual pipeline and the BERTopic model are perfectly aligned. The document sets have been clustered in exactly the same way, with only the cluster IDs for clusters 2 and 3 being swapped. This confirms that cluster IDs in these algorithms are completely arbitrary.

## Visualization & Sensitivity Check

In [ ]:
from umap import UMAP

# A separate 2D UMAP just for plotting -- the 5D one above (umap_model) is fit
# for clustering, not for a readable 2D scatter plot, and its first two columns
# are not the same thing as a proper standalone 2D UMAP projection
umap_2d = UMAP(
    n_components=config.UMAP_N_COMPONENTS_2D,
    min_dist=config.UMAP_MIN_DIST,
    metric=config.UMAP_METRIC,
    random_state=config.UMAP_SEED,
)
reduced_embeddings_2d = umap_2d.fit_transform(embeddings)

titles = books_df['title'].tolist()

fig = topic_model.visualize_documents(
    titles,
    reduced_embeddings=reduced_embeddings_2d,
    width=1200,
    hide_annotations=True
)

fig.update_layout(font=dict(size=16))

fig.show()

In [ ]:
topic_sizes = final_topic_summary[final_topic_summary['Topic'] != -1]

topic_sizes.plot.bar(
    x='Human_Label',
    y='Count',
    color='mediumseagreen',
    legend=False,
    figsize=(10, 6)
)

plt.title('Topic Sizes (Number of Books per Topic)')
plt.xlabel('Discovered Topics')
plt.ylabel('Document Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
topics_to_inspect = [0, 2, 4]

for t in topics_to_inspect:
    print(f"\n{'='*60}")
    print(f"Inspecting Topic {t}")
    print(f"{'='*60}")

    sample_docs = books_df[books_df['Topic'] == t].sample(n=5, random_state=config.INSPECTION_SEED)

    for idx, row in sample_docs.iterrows():
        print(f"🔸 {row['summary'][:300]}...\n")

Topic Inspection Notes
Based on the manual inspection of the representative documents for the selected topics, here is the qualitative assessment:

Topic 4 (War & Military History): This cluster is highly coherent. The sampled documents share a very strong semantic focus on war, combat, and military history, explicitly mentioning World War I, World War II, Vietnam, and the Civil War.

Topic 2 (Fiction, Romance & Sci-Fi): This topic is also coherent. While it spans a few subgenres (fantasy, romance, suspense, and sci-fi), it successfully groups together narrative fiction and novels. The underlying semantic connection is clearly storytelling and fictional literature.

Topic 0 (Reference, Educational & Instructional): This cluster is mixed and overly broad. The model has grouped entirely unrelated subjects together, including biblical studies, a software tutorial (Flash MX), a fraud prevention guide, and a Yoruba language textbook. Rather than a semantic connection, this cluster appears to be a catch-all for instructional, academic, or reference materials, capturing stylistic artifacts rather than a unified theme.

Conclusion:
The BERTopic model successfully isolates distinct, coherent genres like military history and narrative fiction. However, it struggles with non-fiction and educational texts, grouping them into a mixed "catch-all" cluster based on their instructional tone rather than their actual subject matter.

In [ ]:
# Run exactly one sensitivity experiment: only min_cluster_size changes
sensitivity_result = topics_mod.sensitivity_check(
    documents, embeddings, embedding_model, umap_model, baseline_topics=topics
)

print(f"Baseline (min_cluster_size={config.MIN_CLUSTER_SIZE}) -> "
      f"Clusters: {sensitivity_result['baseline_clusters']} | Outliers: {sensitivity_result['baseline_outliers']}")
print(f"New Run  (min_cluster_size={config.SENSITIVITY_MIN_CLUSTER_SIZE})-> "
      f"Clusters: {sensitivity_result['new_clusters']} | Outliers: {sensitivity_result['new_outliers']}")
print(f"Baseline Outlier Ratio: {sensitivity_result['baseline_outlier_ratio']:.2%}")
print(f"New Outlier Ratio: {sensitivity_result['new_outlier_ratio']:.2%}")

### Sensitivity Check & Conclusion

**Experiment:**
To test the sensitivity of the clustering pipeline, the `min_cluster_size` parameter in the HDBSCAN model was increased from 5 to 10, keeping all other parameters constant.

**Before/After Comparison:**
* Baseline (min_cluster_size=5): 6 Clusters, 39 Outliers
* New Run (min_cluster_size=10): 2 Clusters, 41 Outliers (18.98% Outlier Ratio)

**Qualitative Coherence & Conclusion:**
The discovered structure is highly sensitive to the `min_cluster_size` parameter. By increasing the threshold to 10, the total number of clusters collapsed from 6 to just 2. For a small dataset of ~200 books, requiring at least 10 books to form a topic is too restrictive. It forces the model to either merge distinct genres into overly broad, meaningless mega-clusters or discard them into the noise category.

Therefore, the original structure (`min_cluster_size=5`) is much more meaningful and coherent, successfully capturing the granular semantic diversity (e.g., separating thrillers from military history) without losing too much data to outliers.

(Percentages above are from the original run; rerun the cell above to confirm on a fresh run.)

## Joining Sentiment and Topics

In [ ]:
# Roll excerpt-level sentiment up to one row per book
book_sentiment = analysis_mod.aggregate_book_sentiment(sample_df)

print("--- Aggregated Book-Level Sentiment ---")
display(book_sentiment.head())

In [ ]:
# Join book metadata + sentiment aggregates + topic labels into one analysis table
book_analysis = analysis_mod.join_topics_and_sentiment(books_df, book_sentiment, final_topic_summary)

display(book_analysis.head())

In [ ]:
summary_tables = analysis_mod.create_summary_tables(book_analysis)

# Which topics skew most positive on average?
display(summary_tables["topic_pos_sentiment"])

In [ ]:
# Which topics tend to have the longest or shortest summaries?
display(summary_tables["topic_length"])

In [ ]:
# Which books fall outside every discovered topic (topical outliers)?
display(summary_tables["topical_outliers"].head())

## Saving Outputs

Save metrics and topic tables to `outputs/` so results can be inspected
without rerunning the full pipeline.

In [ ]:
import json
from pathlib import Path

Path(config.METRICS_DIR).mkdir(parents=True, exist_ok=True)
Path(config.TOPICS_DIR).mkdir(parents=True, exist_ok=True)

metrics_out = {
    "macro_precision": sentiment_eval["macro_precision"],
    "macro_recall": sentiment_eval["macro_recall"],
    "macro_f1": sentiment_eval["macro_f1"],
    "baseline_vs_transformer": {
        "baseline_macro_f1": comparison["baseline"]["macro_f1"],
        "transformer_macro_f1": comparison["transformer"]["macro_f1"],
    },
    "sensitivity_check": {
        "baseline_clusters": sensitivity_result["baseline_clusters"],
        "baseline_outliers": sensitivity_result["baseline_outliers"],
        "baseline_outlier_ratio": sensitivity_result["baseline_outlier_ratio"],
        "new_clusters": sensitivity_result["new_clusters"],
        "new_outliers": sensitivity_result["new_outliers"],
        "new_outlier_ratio": sensitivity_result["new_outlier_ratio"],
    },
}
with open(f"{config.METRICS_DIR}/metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

final_topic_summary.drop(columns=["Representative_Docs"], errors="ignore").to_csv(
    f"{config.TOPICS_DIR}/topic_summary.csv", index=False
)
book_analysis.to_csv(f"{config.METRICS_DIR}/book_analysis.csv", index=False)

print("Saved outputs/metrics/metrics.json, outputs/metrics/book_analysis.csv, outputs/topics/topic_summary.csv")